# The two training levers: MTHSPL data and a strength head (Colab)

The follow-up to the write-up's §7. One W&B grid (`sweeps/levers.yaml`) runs the final recipe (SapBERT + ingredient negatives + strength normalizer) with and without the FDA label names (MTHSPL, license level 0) as extra training strings, and with and without an auxiliary strength-classification head, on each of the six ingredient splits of §5.8 and two training seeds: 2 × 2 × 6 × 2 = 48 runs. Every contrast is a difference within one (split, seed) pair. Two tiny control sweeps (`sweeps/levers_control_steps.yaml`, `sweeps/levers_control_size.yaml`, 2 runs each) check whether an MTHSPL gain is just more optimizer steps or just more rows.

**Before running:** Runtime → Change runtime type → **A100 GPU**. A VANDF-only run takes about two minutes, an MTHSPL run about eight (4.2× the training rows); the whole grid is roughly four hours, so expect two sessions. The sweeps are registered locally first (`uv run scripts/08_sweep.py --create --config sweeps/levers.yaml`); paste the ids into cell 5.

A grid sweep hands each combination out once. If a run crashes, rerun the cell: the agent picks up the untried cells, and `13_levers.py summarize` reports any cell still missing rather than failing.

## 1. Check the GPU
Expect an A100. If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
A plain clone of the public repo. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
BRANCH = "levers"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone directory goes on `sys.path`, so `import rxnorm_vandf` reads the code straight from the clone. The pip line adds only what Colab doesn't already ship, without upgrading what it does.

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, numpy, pandas, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets (key icon, `WANDB_API_KEY`, *Notebook access* on). `wandb.login()` reads the environment variable, so nothing is pasted or printed.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Start the agent
`run_one` builds a `TrainConfig` with the fixed choices (data from the artifact named in the sweep config, no model artifact per trial) and passes `sweep=True`, so `train()` takes this trial's parameters, including `train_sources`, `aux`, `dataset_subdir` and `seed`, from `wandb.config`. Run the main grid first; the two control sweeps take about 25 minutes together.

Order: `levers` (the remaining cells of the main grid), then `rerun` (the aux-off cells on v3 to v6: twelve failed in the first session on a loader assertion, fixed since, and two v6 cells were killed with the agent; a grid sweep never re-issues a failed cell, and one of the sixteen is a duplicate of a finished cell), then the two controls. `13_levers.py summarize` merges the main and rerun sweeps.

In [ ]:
import gc
from rxnorm_vandf.train import TrainConfig, train

SWEEPS = {
    "levers": "kettle-labs/rxnorm-vandf/nypttmi8",          # sweeps/levers.yaml, 48 runs
    "rerun": "kettle-labs/rxnorm-vandf/ad9nakej",           # sweeps/levers_rerun.yaml: the 14 cells that failed or were killed in the first session (+1 duplicate)
    "control-steps": "kettle-labs/rxnorm-vandf/2exoct4h",   # sweeps/levers_control_steps.yaml, 2 runs
    "control-size": "kettle-labs/rxnorm-vandf/ocmpf5na",    # sweeps/levers_control_size.yaml, 2 runs
}
RUN = "levers"

def run_one():
    train(TrainConfig(data_dir=None, output_dir="models", log_model=False,
                      tags=["sweep", "colab", RUN]), sweep=True)
    gc.collect(); torch.cuda.empty_cache()

wandb.agent(SWEEPS[RUN], function=run_one, count={"levers": 48, "rerun": 16}.get(RUN, 2))

## 6. Afterwards
On the sweep page, group the runs table by `train_sources` and `aux`. Back on the local machine, `uv run scripts/13_levers.py summarize --sweep <id> --steps <id> --size <id>` writes `outputs/levers/levers.md`: the 2 × 2 per split and seed, the two main effects and their interaction with 95% intervals over the twelve (split, seed) units, the per-component accuracies, the MTHSPL transfer accuracy of the VANDF-only arm, and the two controls.